# Explore CheXpert Labels

Goal: understand actual label file structure (columns, uncertain-label representation, class balance) before writing `parse_chexpert_labels()`.
Use a small local sample only - do not sync the full dataset locally.

In [2]:
import pandas as pd

In [3]:
import json

with open('/Users/runosiakpebru/Downloads/findings_fixed.json') as f:
    data = [json.loads(line) for line in f]

df_findings_fixed = pd.DataFrame(data)
df_findings_fixed.head()

,path_to_image,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices,No Finding
0,train/patient42142/study5/view1_frontal.jpg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
1,train/patient42142/study8/view1_frontal.jpg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
2,train/patient42142/study2/view1_frontal.jpg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
3,train/patient42142/study4/view1_frontal.jpg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
4,train/patient42142/study3/view1_frontal.jpg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0


In [4]:
with open('/Users/runosiakpebru/Downloads/report_fixed.json') as f:
    data = [json.loads(line) for line in f]

df_report_fixed = pd.DataFrame(data)
df_report_fixed.head(3)

,path_to_image,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices,No Finding
0,train/patient42142/study5/view1_frontal.jpg,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,1.0,NaN
1,train/patient42142/study8/view1_frontal.jpg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,NaN,NaN,NaN,1.0,NaN
2,train/patient42142/study2/view1_frontal.jpg,NaN,NaN,1.0,NaN,NaN,NaN,NaN,1.0,NaN,1.0,NaN,NaN,1.0,NaN


In [5]:
label_cols = [c for c in df_findings_fixed.columns if c != 'path_to_image']

def unique_label_values(df, label_cols):
    return sorted(
        pd.unique(df[label_cols].values.ravel()),
        key=lambda x: (pd.isna(x), x)  # push NaN to the end, keep numbers ordered
    )

print("findings:", unique_label_values(df_findings_fixed, label_cols))
print("report:  ", unique_label_values(df_report_fixed,   label_cols))

findings: [np.float64(-1.0), np.float64(0.0), np.float64(1.0), np.float64(nan)]
report:   [np.float64(-1.0), np.float64(0.0), np.float64(1.0), np.float64(nan)]


In [6]:
def label_balance(df, label_cols):
    n = len(df)
    out = pd.DataFrame({
        'pos_1':      (df[label_cols] == 1.0).sum(),
        'neg_0':      (df[label_cols] == 0.0).sum(),
        'uncertain':  (df[label_cols] == -1.0).sum(),
        'blank_nan':  df[label_cols].isna().sum(),
    })
    out['total']    = n
    out['pos_rate'] = (out['pos_1'] / n).round(4)   # positives as a fraction of all rows
    return out.sort_values('pos_1', ascending=False)

print("=== findings_fixed ===")
print(label_balance(df_findings_fixed, label_cols))
print("\n=== report_fixed ===")
print(label_balance(df_report_fixed, label_cols))

=== findings_fixed ===
                             pos_1  neg_0  uncertain  blank_nan   total  \
No Finding                  167933      0          0      55529  223462   
Lung Opacity                 32358   1488        105     189511  223462   
Support Devices              31710    377          1     191374  223462   
Pleural Effusion             24511  10586       2281     186084  223462   
Edema                        12148   4443       2687     204184  223462   
Cardiomegaly                 10971  11261       1554     199676  223462   
Atelectasis                   9136    167       8371     205788  223462   
Pneumothorax                  6391  18577        702     197792  223462   
Consolidation                 4159   7083       5938     206282  223462   
Enlarged Cardiomediastinum    4130  12928       6909     199495  223462   
Lung Lesion                   3490    329        459     219184  223462   
Fracture                      3266   2750        148     217298  223462   
Pl

In [7]:
pd.Series.value_counts(df_report_fixed['No Finding'])

No Finding
1.0    13053
Name: count, dtype: int64

In [8]:
finding_cols = [
    'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
    'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia',
    'Atelectasis', 'Pneumothorax', 'Pleural Effusion',
    'Pleural Other', 'Fracture', 'Support Devices', 'No Finding'
]

# value_counts per column, including NaN, side by side
counts = df_report_fixed[finding_cols].apply(lambda col: col.value_counts(dropna=False))
counts

,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices,No Finding
-1.0,19327,7023,550,1361,13032,28101,23045,35267,2878,7472,2971,550,34,NaN
0.0,32721,25996,6485,1473,23071,33262,4720,745,67218,43956,50,4594,2279,NaN
1.0,9141,36172,110580,13149,54255,14709,5963,38283,19136,94076,5760,11143,133282,13053.0
NaN,162273,154271,105847,207479,133104,147390,189734,149167,134230,77958,214681,207175,87867,210409.0


In [9]:
proportions = df_report_fixed[finding_cols].apply(lambda col: col.value_counts(dropna=False, normalize=True))
proportions.round(3)

,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices,No Finding
-1.0,0.086,0.031,0.002,0.006,0.058,0.126,0.103,0.158,0.013,0.033,0.013,0.002,0.000,NaN
0.0,0.146,0.116,0.029,0.007,0.103,0.149,0.021,0.003,0.301,0.197,0.000,0.021,0.010,NaN
1.0,0.041,0.162,0.495,0.059,0.243,0.066,0.027,0.171,0.086,0.421,0.026,0.050,0.596,0.058
NaN,0.726,0.690,0.474,0.928,0.596,0.660,0.849,0.668,0.601,0.349,0.961,0.927,0.393,0.942


In [10]:
no_finding_rows = df_report_fixed[df_report_fixed['No Finding'] == 1.0]

other_cols = [c for c in finding_cols if c != 'No Finding']

# For each of the 13 other findings, count how many "No Finding" rows
# also have a positive (1.0) label - should be zero if the data is internally consistent
contradiction_counts = (no_finding_rows[other_cols] == 1.0).sum()
contradiction_counts

Enlarged Cardiomediastinum       8
Cardiomegaly                     0
Lung Opacity                    19
Lung Lesion                     11
Edema                            6
Consolidation                    2
Pneumonia                        0
Atelectasis                     85
Pneumothorax                     0
Pleural Effusion                 7
Pleural Other                    4
Fracture                         0
Support Devices               6012
dtype: int64

In [11]:
uncertain_counts = (no_finding_rows[other_cols] == -1.0).sum()
uncertain_counts

Enlarged Cardiomediastinum    8
Cardiomegaly                  6
Lung Opacity                  6
Lung Lesion                   1
Edema                         3
Consolidation                 6
Pneumonia                     3
Atelectasis                   5
Pneumothorax                  0
Pleural Effusion              4
Pleural Other                 0
Fracture                      0
Support Devices               8
dtype: int64

In [12]:
df_chexpert_plus_240401= pd.read_csv('/Users/runosiakpebru/Downloads/df_chexpert_plus_240401.csv')


In [13]:
df_chexpert_plus_240401.columns.to_list()

['path_to_image',
 'path_to_dcm',
 'frontal_lateral',
 'ap_pa',
 'deid_patient_id',
 'patient_report_date_order',
 'report',
 'section_narrative',
 'section_clinical_history',
 'section_history',
 'section_comparison',
 'section_technique',
 'section_procedure_comments',
 'section_findings',
 'section_impression',
 'section_end_of_impression',
 'section_summary',
 'section_accession_number',
 'age',
 'sex',
 'race',
 'ethnicity',
 'interpreter_needed',
 'insurance_type',
 'recent_bmi',
 'deceased',
 'split']

In [14]:
df_chexpert_plus_240401.head(3)

,path_to_image,path_to_dcm,frontal_lateral,ap_pa,deid_patient_id,patient_report_date_order,report,section_narrative,section_clinical_history,section_history,...,section_accession_number,age,sex,race,ethnicity,interpreter_needed,insurance_type,recent_bmi,deceased,split
0,train/patient00003/study1/view1_frontal.jpg,train/patient00003/study1/view1_frontal.dcm,Frontal,AP,patient00003,1,"NARRATIVE:\nCHEST, ONE VIEW: 2-10-2001\nFINDIN...","\nCHEST, ONE VIEW: 2-10-2001\n",NaN,NaN,...,\nLFEWOZWVDRX\nThis report has been anonymized...,41.0,Male,White,Non-Hispanic/Non-Latino,Unknown,Private Insurance,NaN,No,train
1,train/patient00007/study2/view1_frontal.jpg,train/patient00007/study2/view1_frontal.dcm,Frontal,AP,patient00007,2,NARRATIVE:\nChest 1 View: July 20\n \nHISTORY:...,\nChest 1 View: July 20\n \n,NaN,"Male, 69 years old, intubated.\n \n",...,\n485K2I4588\nThis report has been anonymized....,69.0,Male,Other,Hispanic/Latino,No,Private Insurance,NaN,No,train
2,train/patient00007/study1/view1_frontal.jpg,train/patient00007/study1/view1_frontal.dcm,Frontal,AP,patient00007,1,NARRATIVE:\nChest 1 View: 12-28-2000\n \nHISTO...,\nChest 1 View: 12-28-2000\n \n,NaN,"Male, 69 years old, Check tube placement.\n \n",...,\nRN\nThis report has been anonymized. All dat...,69.0,Male,Other,Hispanic/Latino,No,Private Insurance,NaN,No,train


In [15]:
df_chexpert_plus_240401.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 223462 entries, 0 to 223461
Data columns (total 27 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   path_to_image               223462 non-null  object 
 1   path_to_dcm                 223462 non-null  object 
 2   frontal_lateral             223462 non-null  object 
 3   ap_pa                       191071 non-null  object 
 4   deid_patient_id             223462 non-null  object 
 5   patient_report_date_order   223462 non-null  int64  
 6   report                      223462 non-null  object 
 7   section_narrative           223434 non-null  object 
 8   section_clinical_history    138866 non-null  object 
 9   section_history             38192 non-null   object 
 10  section_comparison          213125 non-null  object 
 11  section_technique           10732 non-null   object 
 12  section_procedure_comments  25669 non-null   object 
 13  section_findin

In [16]:
df_filtered_chexpert=df_chexpert_plus_240401[['path_to_image', 'section_impression', 'section_findings', 'split']]
df_filtered_chexpert.head(3)

,path_to_image,section_impression,section_findings,split
0,train/patient00003/study1/view1_frontal.jpg,\n1. NO EVIDENCE OF PNEUMOTHORAX.\n2. MILD INT...,"Costophrenic angles sharp, without evidence o...",train
1,train/patient00007/study2/view1_frontal.jpg,\n \nLow lung volumes. Stable moderate enlar...,NaN,train
2,train/patient00007/study1/view1_frontal.jpg,"\n \nLow lung volumes, and overlying trauma b...",NaN,train


In [17]:
merged = df_report_fixed.merge(df_filtered_chexpert, on='path_to_image', how='outer', indicator=True)
merged['_merge'].value_counts()

_merge
both          223462
left_only          0
right_only         0
Name: count, dtype: int64

In [18]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 223462 entries, 0 to 223461
Data columns (total 19 columns):
 #   Column                      Non-Null Count   Dtype   
---  ------                      --------------   -----   
 0   path_to_image               223462 non-null  object  
 1   Enlarged Cardiomediastinum  61189 non-null   float64 
 2   Cardiomegaly                69191 non-null   float64 
 3   Lung Opacity                117615 non-null  float64 
 4   Lung Lesion                 15983 non-null   float64 
 5   Edema                       90358 non-null   float64 
 6   Consolidation               76072 non-null   float64 
 7   Pneumonia                   33728 non-null   float64 
 8   Atelectasis                 74295 non-null   float64 
 9   Pneumothorax                89232 non-null   float64 
 10  Pleural Effusion            145504 non-null  float64 
 11  Pleural Other               8781 non-null    float64 
 12  Fracture                    16287 non-null   float64 
 13 

In [19]:
merged.isnull().sum()

path_to_image                      0
Enlarged Cardiomediastinum    162273
Cardiomegaly                  154271
Lung Opacity                  105847
Lung Lesion                   207479
Edema                         133104
Consolidation                 147390
Pneumonia                     189734
Atelectasis                   149167
Pneumothorax                  134230
Pleural Effusion               77958
Pleural Other                 214681
Fracture                      207175
Support Devices                87867
No Finding                    210409
section_impression               145
section_findings              163993
split                              0
_merge                             0
dtype: int64

In [20]:
merged.columns.to_list()

['path_to_image',
 'Enlarged Cardiomediastinum',
 'Cardiomegaly',
 'Lung Opacity',
 'Lung Lesion',
 'Edema',
 'Consolidation',
 'Pneumonia',
 'Atelectasis',
 'Pneumothorax',
 'Pleural Effusion',
 'Pleural Other',
 'Fracture',
 'Support Devices',
 'No Finding',
 'section_impression',
 'section_findings',
 'split',
 '_merge']

In [21]:
merged = merged.drop(columns=['section_findings'])

In [22]:
merged=merged.dropna(subset=['section_impression'])

In [23]:
merged['split'].value_counts()

split
train    223083
valid       234
Name: count, dtype: int64

In [24]:
merged.shape

(223317, 18)

In [25]:
merged['patient_id']=merged['path_to_image'].str.split('/').str[1]
merged.head(3)

,path_to_image,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices,No Finding,section_impression,split,_merge,patient_id
0,train/patient00001/study1/view1_frontal.jpg,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,\n \n1. Left subclavian central venous cathet...,train,both,patient00001
1,train/patient00002/study1/view1_frontal.jpg,NaN,0.0,1.0,NaN,NaN,NaN,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,\n1. MARKED KYPHOSIS WITH STATUS POST VERTEBRO...,train,both,patient00002
2,train/patient00002/study1/view2_lateral.jpg,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0.0,NaN,1.0,\n1. MARKED KYPHOSIS WITH STATUS POST VERTEBRO...,train,both,patient00002


In [26]:
unique=merged['patient_id'].unique()
unique

array(['patient00001', 'patient00002', 'patient00003', ...,
       'patient64738', 'patient64739', 'patient64740'],
      shape=(64706,), dtype=object)

In [27]:
from sklearn.model_selection import train_test_split
train_ids, remaining_ids = train_test_split(unique, test_size=0.10, random_state=42)
val_ids, test_ids = train_test_split(remaining_ids, test_size=0.50, random_state=42)
train_ids, val_ids, test_ids = set(train_ids), set(val_ids), set(test_ids)
def assign_split(pid):
    if pid in train_ids:
        return 'train'
    elif pid in val_ids:
        return 'val'
    else:
        return 'test'

merged['split_custom'] = merged['patient_id'].apply(assign_split)
merged['split_custom'].value_counts()

split_custom
train    201215
test      11127
val       10975
Name: count, dtype: int64

In [28]:
set(merged[merged.split_custom=='train'].patient_id) & set(merged[merged.split_custom=='val'].patient_id)

set()

In [29]:
merged.columns.to_list()

['path_to_image',
 'Enlarged Cardiomediastinum',
 'Cardiomegaly',
 'Lung Opacity',
 'Lung Lesion',
 'Edema',
 'Consolidation',
 'Pneumonia',
 'Atelectasis',
 'Pneumothorax',
 'Pleural Effusion',
 'Pleural Other',
 'Fracture',
 'Support Devices',
 'No Finding',
 'section_impression',
 'split',
 '_merge',
 'patient_id',
 'split_custom']

In [30]:
merged = merged.drop(columns=['split', '_merge'])
clean = merged.rename(columns={'split_custom': 'split'})

In [31]:
import numpy as np
pathology_cols = [c for c in finding_cols if c != 'No Finding']
pathology_cols
targets = clean[pathology_cols].fillna(0.0)
mask = (targets != -1.0).astype(float)
targets = targets.replace(-1.0, 0.0)

In [32]:
pathology_only = [c for c in pathology_cols if c != 'Support Devices']  # 12 true pathology columns

no_finding = clean['No Finding'].fillna(0.0)
print(targets.min().min(), targets.max().max())  # should print 0.0 1.0 - no -1s or NaNs left in targets
print(mask.min().min(), mask.max().max())          # should print 0.0 1.0 - confirms mask is properly binary
print((mask == 0).sum())                            # per-column count of masked (uncertain) cells - should roughly match your earlier -1.0 counts

0.0 1.0
0.0 1.0
Enlarged Cardiomediastinum    19317
Cardiomegaly                   7018
Lung Opacity                    550
Lung Lesion                    1360
Edema                         13024
Consolidation                 28085
Pneumonia                     23021
Atelectasis                   35243
Pneumothorax                   2877
Pleural Effusion               7471
Pleural Other                  2969
Fracture                        549
Support Devices                  34
dtype: int64


In [33]:
clean.tail(3)

,path_to_image,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices,No Finding,section_impression,patient_id,split
223459,valid/patient64738/study1/view1_frontal.jpg,NaN,0.0,1.0,NaN,NaN,-1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,\n \n1.INTERVAL PLACEMENT OF A LEFT UPPER EX...,patient64738,train
223460,valid/patient64739/study1/view1_frontal.jpg,NaN,1.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,\n \nCARDIAC SILHOUETTE REMAINS MILDLY ENLARGE...,patient64739,train
223461,valid/patient64740/study1/view1_frontal.jpg,NaN,-1.0,1.0,NaN,1.0,NaN,-1.0,-1.0,0.0,1.0,NaN,NaN,1.0,NaN,\n1. THERE HAS BEEN INTERVAL INCREASE ...,patient64740,train


In [34]:
clean.to_parquet('../data/chexpert_plus_clean.parquet', index=False)

In [35]:
train_set=clean[(clean['split'] == 'train')]

In [36]:
src_train=train_set['section_impression'].values.tolist()

In [37]:
train_stmts=[]
for stmt in src_train:
    line=stmt.replace('\n'," ").lower().strip()
    train_stmts.append(line.split(','))


In [38]:
import re

PUNCT_TO_KEEP = ['.', ',', '(', ')', ':', '/', '-']

def tokenize(text):
    text = text.lower().strip()
    text = text.replace('\n', ' ')

    # pad each punctuation char with spaces so it splits off as its own token
    for p in PUNCT_TO_KEEP:
        text = text.replace(p, f' {p} ')

    # collapse multiple spaces down to one
    text = re.sub(r'\s+', ' ', text).strip()

    return text.split(' ')

#tokenize('1. no evidence of pneumothorax. 2. mild interstitial pulmonary edema.')

In [39]:
train_tokens= [tokenize(t) for t in src_train]
print(f"Total sentence :{len(train_tokens)}")
for i in range(5):
    print(f"Text: {train_tokens[i]}")

Total sentence :201215
Text: ['1', '.', 'left', 'subclavian', 'central', 'venous', 'catheter', 'with', 'tip', 'in', 'superior', 'vena', 'cava', '.', 'no', 'evidence', 'for', 'pneumothorax', '.', '"physician', 'to', 'physician', 'radiology', 'consult', 'line', ':', '(', '248', ')', '305', '-', '8884"']
Text: ['1', '.', 'marked', 'kyphosis', 'with', 'status', 'post', 'vertebroplasty', 'changes', 'in', 'the', 'mid', '-', 'thoracic', 'region', '.', 'chronicity', 'of', 'the', 'multiple', 'wedge', 'deformities', 'and', 'compression', 'fractures', 'are', 'age', 'indeterminate', 'in', 'the', 'absence', 'of', 'a', 'comparison', 'study', '.', '2', '.', 'patchy', 'air', 'space', 'opacities', 'in', 'the', 'left', 'lower', 'lobe', 'posteriorly', 'concerning', 'for', 'consolidation', '.']
Text: ['1', '.', 'marked', 'kyphosis', 'with', 'status', 'post', 'vertebroplasty', 'changes', 'in', 'the', 'mid', '-', 'thoracic', 'region', '.', 'chronicity', 'of', 'the', 'multiple', 'wedge', 'deformities', 'and'